In [101]:
import logging
from random import random, randrange, randint

import torch

In [102]:
logger = logging.Logger(__name__)
logger.setLevel(logging.INFO)

In [103]:
class SnakeSolver(torch.nn.Module):
    def __init__(self, square_side_size: int = 5) -> None:
        super().__init__()
        
        self.data = torch.nn.Sequential(
            torch.nn.Linear(square_side_size * square_side_size, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 4)
        )
    
    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return self.data(input)

In [104]:
def get_device() -> torch.device:
    device_name: str = "cuda" if torch.cuda.is_available() else "cpu"

    logger.info(f"Device name: {device_name}")

    return torch.device(device_name)

In [105]:
def select_action(model: SnakeSolver, state: torch.Tensor, action_size: int, epsilon: float) -> int:
    if random() < epsilon:
        return randrange(action_size)
    
    with torch.no_grad():
        result: torch.Tensor = model(state)
        return int(result.argmax(dim=1).item())

In [106]:
from SnakeGame.Board import Board

def get_random_board() -> Board:
    width: int = randint(6, 25)
    height: int = randint(6, 25)

    if not (width * height) % 2:
        width += 1

    return Board(width, height)

In [107]:
from collections import deque
from dataclasses import dataclass, field
from random import sample


@dataclass
class StateSaver:
    state: torch.Tensor
    action: int
    reward: float
    next_state: torch.Tensor | None
    done: bool


class Buffer:
    def __init__(self, capacity: int) -> None:
        self.data = deque(maxlen=capacity)
    
    def append(self, value: StateSaver) -> None:
        self.data.append(value)

    def sample(self, batch_size: int) -> list[StateSaver]:
        return sample(self.data, batch_size)

    def __len__(self) -> int:
        return len(self.data)

In [108]:
@dataclass
class EpsilonData:
    epsilon_start: float = 1.0
    current: float = 1.0
    epsilon_end: float = 0.05
    epsilon_decay: float = 0.995


@dataclass
class TrainData:
    device: torch.device
    model: SnakeSolver
    target_model: SnakeSolver
    optimizer: torch.optim.Optimizer
    transition_buffer: Buffer
    
    gamma: float = 0.95
    batch_size: int = 32
    epsilon_data: EpsilonData = field(default_factory=lambda : EpsilonData())

In [109]:
from utils import Vector2D


def get_state_from_board(board: Board) -> torch.Tensor:
    def get_new_coords(current: Vector2D) -> tuple[int, int] | None:
        if abs(new_x := (current.x - result_center.x)) > 2 or abs(new_y := (current.y - result_center.y)) > 2:
            return None
        
        return new_x + 2, new_y + 2

    board_x, board_y = board.get_size().to_tuple()

    result = torch.zeros((5, 5), dtype=torch.float32)
    
    snake = board.get_snake()
    result_center = snake[0]
    result[2, 2] = 1.0

    for result_index_x, x_add in enumerate(range(-2, 3)):
        for result_index_y, y_add in enumerate(range(-2, 3)):
            if board_x > result_center.x + x_add >= 0 and board_y > result_center.y + y_add >= 0:
                continue

            result[result_index_x, result_index_y] = -1.0

    for snake_part in snake[1:]:
        if (new_coords := get_new_coords(snake_part)) is None:
            continue
        
        result[new_coords] = 1.0
    
    if (new_coords := get_new_coords(board.get_apple())) is None:
        return result
    
    result[new_coords] = 2.0

    return result

In [110]:
from SnakeGame.Board import BoardStatus


def get_reward(board: Board, length: int) -> tuple[float, int]:
    result: float = -0.5
    
    if (new_length := len(board.get_snake()) > length):
        result += 10.0
    
    if board._status == BoardStatus.WIN:
        result += 300.0
    elif board._status == BoardStatus.LOSE:
        result -= 300.0
    
    return result, new_length

In [111]:
def optimize_model(data: TrainData) -> None:
    if len(data.transition_buffer) < data.batch_size:
        return
    
    transitions = data.transition_buffer.sample(data.batch_size)

    states = torch.stack([transition.state for transition in transitions])
    actions = torch.tensor(
        [transition.action for transition in transitions],
        dtype=torch.long,
    ).unsqueeze(1)
    rewards = torch.tensor(
        [transition.reward for transition in transitions],
        dtype=torch.float32,
    )
    next_states = torch.stack([transition.next_state for transition in transitions])
    dones = torch.tensor(
        [transition.done for transition in transitions],
        dtype=torch.float32,
    )

    current_q_values = data.model(states).gather(1, actions).squeeze(1)

    with torch.no_grad():
        next_q_values = data.target_model(next_states).max(dim=1).values
        target_q_values = rewards + data.gamma * next_q_values * (1.0 - dones)

    loss = torch.nn.functional.smooth_l1_loss(current_q_values, target_q_values)

    data.optimizer.zero_grad()
    loss.backward()
    data.optimizer.step()

In [112]:
from utils import MoveDirection


def run_epoch(data: TrainData) -> None:
    board = get_random_board()
    length: int = 1

    while board.is_running():
        state = get_state_from_board(board).flatten()
        action = select_action(data.model, state.unsqueeze(0), 4, data.epsilon_data.current)

        board.move(MoveDirection(action))

        reward, length = get_reward(board, length)

        next_state: torch.Tensor = torch.zeros_like(state)
        done: bool = False

        if board.get_status() == BoardStatus.RUNNING:
            next_state = get_state_from_board(board).flatten()
        else:
            done = True

        data.transition_buffer.append(StateSaver(
            state=state,
            action=action,
            reward=reward,
            next_state=next_state,
            done=done
        ))
        
        optimize_model(data)
    
    data.epsilon_data.current = max(
        data.epsilon_data.epsilon_end,
        data.epsilon_data.current * data.epsilon_data.epsilon_decay,
    )

In [113]:
from tqdm import tqdm

def epoch_runner(data: TrainData, epoch_number: int) -> None:
    for _ in tqdm(range(epoch_number)):
        run_epoch(data)

In [114]:
def save_model(data: TrainData, epoches: int, path: str | None = None) -> None:
    if path is None:
        path = f"../models/result_{epoches}.pt"
    
    torch.save(data.model.state_dict(), path)

In [115]:
def train(epoch_number: int) -> None:
    device = get_device()
    
    model = SnakeSolver(5).to(device)
    target_model = SnakeSolver(5).to(device)
    target_model.load_state_dict(model.state_dict())

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    buffer = Buffer(50_000)

    train_data = TrainData(device, model, target_model, optimizer, buffer)

    epoch_runner(train_data, epoch_number)

    save_model(train_data, epoch_number)

In [ ]:
train(20_000)

  2%|▏         | 84/5000 [01:04<1:03:04,  1.30it/s]


KeyboardInterrupt: 